# V11 Tri-Expert Fusion — Updated Notebook (Model + XAI)

**Pipeline:** Raw beats / handcrafted features →
`V8` waveform expert (256-channel residual CNN) +
`V9` waveform expert (384-channel residual CNN with TTA) +
`XGBoost` physiology expert (174 handcrafted beat features)
→ Patient-aware 5-fold calibration-time fusion (`meta_logreg` *or* `weighted_grid`)
→ 5-layer XAI (IG, GradCAM, SHAP, Occlusion, Cross-Attention) on the V9 sub-expert
→ Fusion-level XAI (calibration choice, Δ-vs-V9, expert weights).

**Run order:** Cells 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9 (runs folds) → 10 (model plots) → 11–16 (5-layer XAI) → 17–19 (fusion-level XAI) → 20 (XAI summary).

**Resumability:** A single `v11_state.json` is written under `outputs/v11_triexpert_fusion/`. Any fold whose `fold_summary.json` is already on disk is loaded and skipped on re-run. The XAI cells are self-contained — they reload everything from disk if the kernel is restarted.

**Honest 5-fold results (from prior runs):**

| Model | Macro F1 | F1 Std | Accuracy | AUROC |
|---|---:|---:|---:|---:|
| V11 fusion | 0.8818 | 0.0074 | 0.8837 | 0.9634 |
| V9 expert  | 0.8747 | 0.0072 | 0.8769 | 0.9622 |
| V8 expert  | 0.8714 | 0.0045 | 0.8737 | 0.9589 |
| XGBoost    | 0.6221 | 0.0086 | 0.6260 | 0.8028 |

Per-fold fusion choices: fold 1 → `meta_logreg`, fold 2 → `meta_logreg`, folds 3–5 → `weighted_grid`. Gain over V9: ΔF1 = +0.0070, ΔAcc = +0.0068, ΔAUROC = +0.0012.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 1 — SETUP & CONFIGURATION
#
# Imports, fixed notebook paths (V11 uses the same Codex root as V9),
# global constants, and auto-install checks for the two non-standard
# dependencies this notebook needs:
#   - xgboost (for the third expert)
#   - captum  (for IG + GradCAM in the XAI cells)
#
# The auto-install pattern mirrors how V9 handles captum: try import,
# pip install if missing, re-import.
# ═══════════════════════════════════════════════════════════════

from __future__ import annotations

import os, gc, json, math, random, re, sys, time, warnings, subprocess
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Iterable

import joblib
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, roc_curve, auc,
    confusion_matrix, classification_report,
)
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, label_binarize

warnings.filterwarnings("ignore")
%matplotlib inline

# ── Auto-install xgboost if not present (V9-style pattern) ────────────────────
try:
    from xgboost import XGBClassifier
    _xgb_ok = True
except ImportError:
    print("[setup] xgboost not found — installing...", flush=True)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "xgboost", "-q"])
    from xgboost import XGBClassifier
    _xgb_ok = True

# ── Auto-install captum if not present (needed by XAI cells later) ────────────
try:
    from captum.attr import IntegratedGradients, LayerGradCam
    _captum_ok = True
except ImportError:
    print("[setup] captum not found — installing...", flush=True)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "captum", "-q"])
    from captum.attr import IntegratedGradients, LayerGradCam
    _captum_ok = True

# shap is used in the XAI cells too — installed if missing
try:
    import shap
except ImportError:
    print("[setup] shap not found — installing...", flush=True)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "shap", "-q"])
    import shap

# ── PATHS (unchanged from the original V11 script — notebook-safe) ────────────
ROOT             = Path(r"C:\Users\Admin\Desktop\Projects\8th Sem\Codex")
PROJECT_ROOT     = ROOT.parent
BEATS_ROOT       = PROJECT_ROOT / "Preprocessed_Dataset" / "v4_beats"
CHECKPOINT_ROOT  = BEATS_ROOT / "checkpoints"
OUTPUT_ROOT      = ROOT / "outputs" / "v11_triexpert_fusion"
PLOTS_DIR        = OUTPUT_ROOT / "plots"
ASSETS_DIR       = ROOT / "reports" / "assets"

V8_DIR           = CHECKPOINT_ROOT / "v8_consistent"
V9_DIR           = CHECKPOINT_ROOT / "v9_wider"
FEATURES_FILE    = BEATS_ROOT / "rf" / "features_v4.npz"
BEATS_FILE       = BEATS_ROOT / "beats.dat"
META_FILE        = BEATS_ROOT / "beats_meta.npz"

V11_STATE_F      = OUTPUT_ROOT / "v11_state.json"   # single resumability file

for _d in [OUTPUT_ROOT, PLOTS_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

# ── CONSTANTS ─────────────────────────────────────────────────────────────────
CLASS_NAMES   = [
    "Normal/Non-Cardiac",
    "Structural Heart Disease",
    "Arrhythmia & Electrical",
]
N_CLS         = 3
N_LEADS       = 12
N_RR          = 8
BEAT_LEN      = 300
SAMPLING_RATE = 500          # Hz
BEAT_PRE_MS   = 200
BEAT_PRE      = int(BEAT_PRE_MS * SAMPLING_RATE / 1000)   # 100 samples → R-peak at idx 100
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"

# Reproducibility (the per-fold seeds inside the runner add +fold_number)
np.random.seed(42)
torch.manual_seed(42)
random.seed(42)

print(f"✅  V11 Configured")
print(f"    Notebook root : {ROOT}")
print(f"    Output root   : {OUTPUT_ROOT}")
print(f"    Device        : {DEVICE}")
print(f"    xgboost OK    : {_xgb_ok}")
print(f"    captum OK     : {_captum_ok}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2 — UTILITIES, DATACLASSES, DATASET, AUGMENTATION
#
# Small but important supporting layer:
#   - log() : single-line stdout-flushing print used everywhere
#   - set_seed() : per-fold reproducibility
#   - Assets / FoldIndices / ExpertMetrics : typed containers so the
#     orchestrator code stays readable
#   - BeatRRDataset : memmap-backed dataset, no whole-array copy
#   - augment_beat : same augmentation used in V8/V9 training and in V9's TTA
# ═══════════════════════════════════════════════════════════════

def log(message: str) -> None:
    print(message, flush=True)


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


@dataclass
class Assets:
    beats: np.memmap
    labels: np.ndarray
    rec_ids: np.ndarray
    rr_norm: np.ndarray
    features: np.ndarray
    unique_recs: np.ndarray
    rec_labels: np.ndarray
    record_to_indices: dict
    outer_splits: list


@dataclass
class FoldIndices:
    fold_name: str
    train_idx: np.ndarray
    calib_idx: np.ndarray
    test_idx: np.ndarray
    train_recs: np.ndarray
    calib_recs: np.ndarray
    test_recs: np.ndarray


@dataclass
class ExpertMetrics:
    name: str
    f1_macro: float
    accuracy: float
    auroc: float


def augment_beat(x: np.ndarray, strength: float = 1.0) -> np.ndarray:
    """Same augmentation used in V8/V9 training and reused for V9 TTA."""
    x = x.copy()
    if np.random.rand() < 0.7:
        x += np.random.normal(0, 0.025 * strength, x.shape).astype(np.float32)
    if np.random.rand() < 0.6:
        x *= np.random.uniform(1.0 - 0.15 * strength, 1.0 + 0.15 * strength)
    if np.random.rand() < 0.35:
        n_drop = np.random.randint(1, 3)
        drops = np.random.choice(N_LEADS, n_drop, replace=False)
        x[:, drops] = 0.0
    if np.random.rand() < 0.4:
        shift = np.random.randint(-15, 15)
        x = np.roll(x, shift, axis=0)
    return x.astype(np.float32)


class BeatRRDataset(Dataset):
    """
    Memmap-backed beat dataset. We pass an `indices` array (the patient-aware
    fold split) and the dataset just lazily reads from beats.dat for each item.
    `augment=True` enables augmentation (used for V9 TTA passes 2..N).
    """
    def __init__(self, indices, beats_mmap, rr_arr, augment=False, tta_strength=1.0):
        self.indices = indices
        self.beats = beats_mmap
        self.rr = rr_arr
        self.augment = augment
        self.tta_strength = tta_strength

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = int(self.indices[i])
        beat = self.beats[idx].copy()
        if self.augment:
            beat = augment_beat(beat, strength=self.tta_strength)
        beat_t = torch.tensor(beat.T, dtype=torch.float32)
        rr_t   = torch.tensor(self.rr[idx], dtype=torch.float32)
        return beat_t, rr_t


def natural_epoch_key(path: Path) -> int:
    """Sort checkpoint filenames like v9_fold_1_ep27.pt by their numeric epoch."""
    m = re.search(r"_ep(\d+)\.pt$", path.name)
    return int(m.group(1)) if m else -1


log("✅  Cell 2 utilities ready: log, set_seed, dataclasses, BeatRRDataset, augment_beat")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3 — MODEL DEFINITIONS (V8, V9, shared components)
#
# Three classes shared between V8 and V9:
#   - ResConvBlock         : 1D residual conv (used in both morphology encoders)
#   - RhythmEncoder        : N_RR=8 → 64-d rhythm embedding
#   - CrossAttentionFusion : 2-head cross-attention from morphology → rhythm,
#                            with ._attn saved so the XAI cells can inspect it
#
# Then two model wrappers that differ ONLY in the morphology widths:
#   - DualBranchV8 : MorphologyEncoderV8 (64 → 128 → 256 channels)
#   - DualBranchV9 : MorphologyEncoderV9 (96 → 192 → 384 channels) + TTA at test
#
# These are recreated identically to the V8/V9 training scripts so we can
# load the saved checkpoints without state_dict shape mismatches.
# ═══════════════════════════════════════════════════════════════

class ResConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=7, stride=1):
        super().__init__()
        pad = kernel // 2
        self.c1   = nn.Conv1d(in_ch, out_ch, kernel, stride=stride, padding=pad, bias=False)
        self.bn1  = nn.BatchNorm1d(out_ch)
        self.c2   = nn.Conv1d(out_ch, out_ch, kernel, padding=pad, bias=False)
        self.bn2  = nn.BatchNorm1d(out_ch)
        self.drop = nn.Dropout(0.10)
        if in_ch != out_ch or stride != 1:
            self.skip = nn.Sequential(
                nn.Conv1d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm1d(out_ch),
            )
        else:
            self.skip = nn.Identity()

    def forward(self, x):
        out = F.gelu(self.bn1(self.c1(x)))
        out = self.drop(self.bn2(self.c2(out)))
        return F.gelu(out + self.skip(x))


class RhythmEncoder(nn.Module):
    def __init__(self, rhythm_dim=64):
        super().__init__()
        self.fc1 = nn.Linear(N_RR, rhythm_dim)
        self.bn1 = nn.BatchNorm1d(rhythm_dim)
        self.fc2 = nn.Linear(rhythm_dim, rhythm_dim)
        self.bn2 = nn.BatchNorm1d(rhythm_dim)
        self.skip = nn.Linear(N_RR, rhythm_dim)

    def forward(self, rr):
        out = F.gelu(self.bn1(self.fc1(rr)))
        out = self.bn2(self.fc2(out))
        return F.gelu(out + self.skip(rr))


class CrossAttentionFusion(nn.Module):
    """
    Two-head cross-attention from morphology query → rhythm key/value.
    The attention map is saved on self._attn after every forward pass so
    the XAI-5 cell can read it without re-running the model.
    """
    def __init__(self, morph_dim, rhythm_dim=64, heads=2):
        super().__init__()
        d_head = morph_dim // heads
        self.heads, self.d_head = heads, d_head
        self.Q = nn.Linear(morph_dim, morph_dim)
        self.K = nn.Linear(rhythm_dim, morph_dim)
        self.V = nn.Linear(rhythm_dim, morph_dim)
        self.proj = nn.Linear(d_head, morph_dim)
        self.norm = nn.LayerNorm(morph_dim)
        self.scale = math.sqrt(d_head)
        self._attn = None

    def forward(self, m, r):
        B = m.size(0)
        q = self.Q(m).view(B, self.heads, self.d_head)
        k = self.K(r).view(B, self.heads, self.d_head)
        v = self.V(r).view(B, self.heads, self.d_head)
        w = F.softmax((q * k).sum(-1) / self.scale, dim=1)
        self._attn = w.detach()
        out = (w.unsqueeze(-1) * v).sum(1)
        return self.norm(m + self.proj(out))


# ─── V8: morphology widths 64 → 128 → 256 ────────────────────────────────────
class MorphologyEncoderV8(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(N_LEADS, 64, 9, padding=4, bias=False),
            nn.BatchNorm1d(64), nn.GELU(),
        )
        self.s1 = ResConvBlock(64, 64, 7, 2)
        self.s2 = ResConvBlock(64, 128, 5, 2)
        self.s3 = ResConvBlock(128, 256, 5, 1)
        self.gap = nn.AdaptiveAvgPool1d(1)

    def forward(self, x):
        return self.gap(self.s3(self.s2(self.s1(self.stem(x))))).squeeze(-1)


class DualBranchV8(nn.Module):
    def __init__(self):
        super().__init__()
        self.morph  = MorphologyEncoderV8()
        self.rhythm = RhythmEncoder(64)
        self.fusion = CrossAttentionFusion(256, 64, 2)
        self.head = nn.Sequential(
            nn.Linear(256 + 64, 256), nn.GELU(), nn.Dropout(0.4),
            nn.Linear(256, 128),       nn.GELU(), nn.Dropout(0.2),
            nn.Linear(128, N_CLS),
        )

    def forward(self, beat, rr):
        m = self.morph(beat); r = self.rhythm(rr)
        return self.head(torch.cat([self.fusion(m, r), r], dim=1))


# ─── V9: morphology widths 96 → 192 → 384 ────────────────────────────────────
class MorphologyEncoderV9(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(N_LEADS, 96, 9, padding=4, bias=False),
            nn.BatchNorm1d(96), nn.GELU(),
        )
        self.s1 = ResConvBlock(96, 96, 7, 2)
        self.s2 = ResConvBlock(96, 192, 5, 2)
        self.s3 = ResConvBlock(192, 384, 5, 1)
        self.gap = nn.AdaptiveAvgPool1d(1)

    def forward(self, x):
        return self.gap(self.s3(self.s2(self.s1(self.stem(x))))).squeeze(-1)


class DualBranchV9(nn.Module):
    def __init__(self):
        super().__init__()
        self.morph  = MorphologyEncoderV9()
        self.rhythm = RhythmEncoder(64)
        self.fusion = CrossAttentionFusion(384, 64, 2)
        self.head = nn.Sequential(
            nn.Linear(384 + 64, 384), nn.GELU(), nn.Dropout(0.4),
            nn.Linear(384, 128),       nn.GELU(), nn.Dropout(0.2),
            nn.Linear(128, N_CLS),
        )

    def forward(self, beat, rr):
        m = self.morph(beat); r = self.rhythm(rr)
        return self.head(torch.cat([self.fusion(m, r), r], dim=1))


# Sanity-check shapes match expected
_v8 = DualBranchV8(); _v9 = DualBranchV9()
with torch.no_grad():
    _o8 = _v8(torch.randn(2, N_LEADS, BEAT_LEN), torch.randn(2, N_RR))
    _o9 = _v9(torch.randn(2, N_LEADS, BEAT_LEN), torch.randn(2, N_RR))
assert _o8.shape == (2, N_CLS) and _o9.shape == (2, N_CLS)
log(f"✅  V8 params: {sum(p.numel() for p in _v8.parameters() if p.requires_grad):,}")
log(f"✅  V9 params: {sum(p.numel() for p in _v9.parameters() if p.requires_grad):,}")
del _v8, _v9


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 4 — DATA LOADING AND FOLD CONSTRUCTION
#
# We use the same patient-aware outer split as V8 and V9 so the V11 fusion
# numbers are directly comparable. Specifically:
#   - StratifiedShuffleSplit(n_splits=5, test_size=0.20, random_state=42)
#     over UNIQUE RECORDS (not beats) → guarantees no patient leakage.
#   - Per fold, we carve a 15% calibration split out of the 80% train side
#     (random_state = fold_number - 1) to fit / select the fusion rule.
#
# Everything ends up in an `Assets` dataclass that the runner shares with
# every helper (memmap, labels, RR features, handcrafted features, record-
# to-beat-index map, and the precomputed 5 outer splits).
# ═══════════════════════════════════════════════════════════════

def load_assets() -> Assets:
    meta    = np.load(META_FILE, allow_pickle=True)
    labels  = meta["labels"].astype(np.int32)
    rec_ids = meta["rec_ids"]
    rr      = meta["rr_feats"].astype(np.float32)
    rr_norm = StandardScaler().fit_transform(rr).astype(np.float32)
    rr_norm = np.nan_to_num(rr_norm, nan=0.0, posinf=0.0, neginf=0.0)

    feat_npz = np.load(FEATURES_FILE, allow_pickle=True)
    features = np.nan_to_num(feat_npz["X"].astype(np.float32),
                             nan=0.0, posinf=0.0, neginf=0.0)

    beats = np.memmap(BEATS_FILE, dtype="float32", mode="r",
                      shape=(len(labels), BEAT_LEN, N_LEADS))

    unique_recs = np.unique(rec_ids)
    rec_labels  = np.array([
        int(np.argmax(np.bincount(labels[rec_ids == r], minlength=N_CLS)))
        for r in unique_recs
    ], dtype=np.int32)
    record_to_indices = {r: np.where(rec_ids == r)[0] for r in unique_recs}

    outer_splitter = StratifiedShuffleSplit(n_splits=5, test_size=0.20, random_state=42)
    outer_splits   = list(outer_splitter.split(unique_recs, rec_labels))

    log(f"  Loaded {len(labels):,} beats from {len(unique_recs):,} records")
    log(f"  Class distribution: {np.bincount(labels)}")
    return Assets(
        beats=beats, labels=labels, rec_ids=rec_ids,
        rr_norm=rr_norm, features=features,
        unique_recs=unique_recs, rec_labels=rec_labels,
        record_to_indices=record_to_indices, outer_splits=outer_splits,
    )


def expand_records(record_ids, record_to_indices):
    return np.concatenate([record_to_indices[str(r)] for r in record_ids]).astype(np.int64)


def make_fold_indices(assets: Assets, fold_number: int) -> FoldIndices:
    outer_train_idx, outer_test_idx = assets.outer_splits[fold_number - 1]
    outer_train_recs   = assets.unique_recs[outer_train_idx]
    outer_test_recs    = assets.unique_recs[outer_test_idx]
    outer_train_labels = assets.rec_labels[outer_train_idx]

    inner_splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.15,
                                            random_state=fold_number - 1)
    inner_tr, inner_ca = next(inner_splitter.split(outer_train_recs, outer_train_labels))
    train_recs = outer_train_recs[inner_tr]
    calib_recs = outer_train_recs[inner_ca]

    return FoldIndices(
        fold_name   = f"fold_{fold_number}",
        train_idx   = expand_records(train_recs, assets.record_to_indices),
        calib_idx   = expand_records(calib_recs, assets.record_to_indices),
        test_idx    = expand_records(outer_test_recs, assets.record_to_indices),
        train_recs  = train_recs,
        calib_recs  = calib_recs,
        test_recs   = outer_test_recs,
    )


log("✅  Cell 4 ready: load_assets, make_fold_indices, expand_records")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 5 — EXPERT INFERENCE HELPERS
#
# Three helpers we re-use across all 5 folds:
#
#   make_loader(...)              : DataLoader for a given index array
#   predict_probabilities(...)    : checkpoint-ensemble inference (no TTA)
#                                   Used directly for V8.
#   predict_probabilities_v9_tta  : adds TTA (1 clean pass + N-1 augmented
#                                   passes), then averages with the ensemble.
#                                   Used for V9 (matches V9 training script).
#
#   load_v8_models / load_v9_models : load all v{8,9}_{fold}_ep*.pt checkpoints
#                                     in epoch order. The V8/V9 zip archives
#                                     hold 3 checkpoints per fold.
# ═══════════════════════════════════════════════════════════════

def make_loader(indices, assets, batch_size, augment=False, tta_strength=0.7):
    ds = BeatRRDataset(indices, assets.beats, assets.rr_norm,
                       augment=augment, tta_strength=tta_strength)
    return DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0)


@torch.no_grad()
def predict_probabilities(models, loader):
    """Average softmax probabilities over a list of (already-loaded) models."""
    prob_sum = None
    for m in models:
        m.eval()
        batch_probs = []
        for beat, rr in loader:
            beat = beat.to(DEVICE); rr = rr.to(DEVICE)
            probs = F.softmax(m(beat, rr), dim=1).cpu().numpy()
            batch_probs.append(probs)
        mp = np.concatenate(batch_probs, axis=0)
        prob_sum = mp.astype(np.float64) if prob_sum is None else prob_sum + mp
    if prob_sum is None:
        raise RuntimeError("No models were given to predict_probabilities.")
    return (prob_sum / len(models)).astype(np.float32)


@torch.no_grad()
def predict_probabilities_v9_tta(models, indices, assets, batch_size, n_tta):
    """V9 inference: 1 clean pass + (n_tta - 1) augmented passes, all ensembled."""
    pass_probs = []
    base_loader = make_loader(indices, assets, batch_size=batch_size, augment=False)
    pass_probs.append(predict_probabilities(models, base_loader))
    for _ in range(max(0, n_tta - 1)):
        aug_loader = make_loader(indices, assets, batch_size=batch_size,
                                 augment=True, tta_strength=0.7)
        pass_probs.append(predict_probabilities(models, aug_loader))
    return np.mean(pass_probs, axis=0).astype(np.float32)


def load_v8_models(fold_name):
    ckpts = sorted(V8_DIR.glob(f"v8_{fold_name}_ep*.pt"), key=natural_epoch_key)
    if not ckpts:
        raise FileNotFoundError(f"No V8 checkpoints found for {fold_name} in {V8_DIR}")
    out = []
    for cp in ckpts:
        m = DualBranchV8().to(DEVICE)
        m.load_state_dict(torch.load(cp, map_location=DEVICE))
        m.eval(); out.append(m)
    return out


def load_v9_models(fold_name):
    ckpts = sorted(V9_DIR.glob(f"v9_{fold_name}_ep*.pt"), key=natural_epoch_key)
    if not ckpts:
        raise FileNotFoundError(f"No V9 checkpoints found for {fold_name} in {V9_DIR}")
    out = []
    for cp in ckpts:
        m = DualBranchV9().to(DEVICE)
        m.load_state_dict(torch.load(cp, map_location=DEVICE))
        m.eval(); out.append(m)
    return out


log("✅  Cell 5 ready: make_loader, predict_probabilities, predict_probabilities_v9_tta, load_v{8,9}_models")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 6 — XGBOOST EXPERT TRAINING
#
# The third expert is a gradient-boosted tree over the 174 handcrafted beat
# features (morphology, RR statistics, cross-lead correlations, fiducial
# points). It is intentionally complementary to the waveform CNNs:
#   - CNN learns shape directly from raw beats → strong on structural
#   - XGBoost reads engineered HRV-style features → strong on rhythm patterns
#     that are well-captured by RMSSD / RR-CV but harder for a 300-sample CNN
#
# Training is fast (no GPU needed) and the model is saved per-fold so that
# the resumability path can skip XGBoost retraining when fold_summary.json
# is already on disk.
# ═══════════════════════════════════════════════════════════════

def train_xgb_expert(fold: FoldIndices, assets: Assets, out_dir: Path, seed: int):
    model = XGBClassifier(
        objective       = "multi:softprob",
        num_class       = N_CLS,
        eval_metric     = "mlogloss",
        n_estimators    = 500,
        learning_rate   = 0.05,
        max_depth       = 8,
        min_child_weight= 2,
        subsample       = 0.85,
        colsample_bytree= 0.80,
        reg_lambda      = 2.0,
        tree_method     = "hist",
        random_state    = seed,
        n_jobs          = max(1, (os.cpu_count() or 4) - 1),
    )
    X_train = assets.features[fold.train_idx]
    y_train = assets.labels[fold.train_idx]
    X_calib = assets.features[fold.calib_idx]
    y_calib = assets.labels[fold.calib_idx]
    model.fit(X_train, y_train, eval_set=[(X_calib, y_calib)], verbose=False)
    joblib.dump(model, out_dir / f"{fold.fold_name}_xgb.pkl")
    return model


def evaluate_predictions(y_true, probs, name):
    preds = probs.argmax(axis=1)
    return ExpertMetrics(
        name     = name,
        f1_macro = float(f1_score(y_true, preds, average="macro", zero_division=0)),
        accuracy = float(accuracy_score(y_true, preds)),
        auroc    = float(roc_auc_score(y_true, probs, multi_class="ovr", average="macro")),
    )


log("✅  Cell 6 ready: train_xgb_expert, evaluate_predictions")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 7 — FUSION LOGIC (grid search + meta-learner)
#
# The novelty of V11. On the held-out CALIBRATION split inside each fold,
# we score four candidates and pick the best:
#
#   1. v8   alone  (just argmax V8 probabilities)
#   2. v9   alone  (just argmax V9 probabilities)
#   3. xgb  alone
#   4. weighted_grid : weighted average of the 3 expert probability vectors,
#                      with weights swept on a 21x21 simplex (step 0.05)
#   5. meta_logreg   : multinomial logistic regression fitted on a feature
#                      vector concatenating the 3 expert prob vectors + per-
#                      expert entropies + per-expert margins (15 features)
#
# Whichever has the highest calibration macro-F1 is used to produce the
# final test predictions. Because the choice is made on CALIBRATION (not
# test) we don't peek at the outer-test labels.
#
# In our 5-fold run: folds 1,2 chose meta_logreg, folds 3,4,5 chose weighted_grid.
# ═══════════════════════════════════════════════════════════════

def prob_entropy(probs):
    eps = 1e-8
    return -(probs * np.log(probs + eps)).sum(axis=1, keepdims=True)


def prob_margin(probs):
    top2 = np.partition(probs, kth=-2, axis=1)[:, -2:]
    return (top2[:, 1] - top2[:, 0]).reshape(-1, 1)


def build_meta_features(v8p, v9p, xgbp):
    return np.concatenate([
        v8p, v9p, xgbp,
        prob_entropy(v8p),  prob_entropy(v9p),  prob_entropy(xgbp),
        prob_margin(v8p),   prob_margin(v9p),   prob_margin(xgbp),
    ], axis=1).astype(np.float32)


def grid_search_fusion(y_true, candidates):
    """Sweep weights on a 21-step simplex over {v8, v9, xgb}."""
    names = list(candidates.keys())
    best_name, best_weights, best_probs, best_f1 = "", {}, None, -1.0
    simplex = np.linspace(0.0, 1.0, 21)
    for w0 in simplex:
        for w1 in simplex:
            w2 = 1.0 - w0 - w1
            if w2 < 0 or w2 > 1:
                continue
            weights = np.array([w0, w1, w2], dtype=np.float32)
            if np.count_nonzero(weights) == 0:
                continue
            probs = sum(weights[i] * candidates[names[i]] for i in range(3))
            f1 = f1_score(y_true, probs.argmax(axis=1), average="macro", zero_division=0)
            if f1 > best_f1:
                best_f1      = float(f1)
                best_name    = "weighted_grid"
                best_weights = {names[i]: float(weights[i]) for i in range(3)}
                best_probs   = probs.astype(np.float32)
    if best_probs is None:
        raise RuntimeError("Grid search produced no candidate (should never happen).")
    return best_name, best_weights, best_probs, best_f1


def fit_meta_learner(meta_train, y_train, meta_test):
    sc = StandardScaler()
    Xtr = sc.fit_transform(meta_train)
    Xte = sc.transform(meta_test)
    model = LogisticRegression(max_iter=1500, class_weight="balanced",
                                C=0.5, solver="lbfgs", random_state=42)
    model.fit(Xtr, y_train)
    probs = model.predict_proba(Xte).astype(np.float32)
    return model, sc, probs


def choose_fusion(calib_labels, calib_probs, test_probs):
    """
    Score the candidates (v8, v9, xgb, weighted_grid, meta_logreg) on the
    calibration split. Return: (chosen_name, params_dict, fused_test_probs,
    calibration_f1_of_chosen).
    """
    meta_calib = build_meta_features(calib_probs["v8"], calib_probs["v9"], calib_probs["xgb"])
    meta_test  = build_meta_features(test_probs["v8"],  test_probs["v9"],  test_probs["xgb"])
    meta_model, meta_scaler, meta_probs_test = fit_meta_learner(meta_calib, calib_labels, meta_test)
    meta_probs_calib = meta_model.predict_proba(meta_scaler.transform(meta_calib)).astype(np.float32)

    candidate_scores = []
    for name, p in calib_probs.items():
        f1 = f1_score(calib_labels, p.argmax(axis=1), average="macro", zero_division=0)
        candidate_scores.append((name, float(f1)))

    w_name, w_weights, _, w_f1 = grid_search_fusion(
        calib_labels,
        {"v8": calib_probs["v8"], "v9": calib_probs["v9"], "xgb": calib_probs["xgb"]},
    )
    w_test = (w_weights["v8"] * test_probs["v8"]
              + w_weights["v9"] * test_probs["v9"]
              + w_weights["xgb"] * test_probs["xgb"]).astype(np.float32)
    candidate_scores.append((w_name, w_f1))

    meta_f1 = f1_score(calib_labels, meta_probs_calib.argmax(axis=1),
                       average="macro", zero_division=0)
    candidate_scores.append(("meta_logreg", float(meta_f1)))

    best_name, _ = max(candidate_scores, key=lambda x: x[1])
    if best_name in calib_probs:
        return best_name, {best_name: 1.0}, test_probs[best_name], dict(candidate_scores)[best_name]
    if best_name == "weighted_grid":
        return best_name, w_weights, w_test, w_f1
    return "meta_logreg", {"uses_meta_features": 1.0}, meta_probs_test, float(meta_f1)


log("✅  Cell 7 ready: choose_fusion, grid_search_fusion, fit_meta_learner, build_meta_features")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 8 — FOLD-RUNNER + SUMMARISE
#
# RESUMABILITY (V9 pattern, single v11_state.json):
#   - v11_state.json sits at OUTPUT_ROOT/v11_state.json
#   - It records {'completed_folds': [...], 'fold_results': {fold_1: {...}, ...}}
#   - run_fold() is called only for folds NOT in completed_folds
#   - If a fold has a `fold_summary.json` on disk but isn't in v11_state.json
#     (e.g. older run before the state file existed), we ingest it into the
#     state file rather than re-running.
#
# Per fold the runner:
#   1. Trains XGBoost on the train split (or loads cached .pkl)
#   2. Runs V8 ensemble inference on calib + test
#   3. Runs V9 TTA ensemble inference on calib + test
#   4. Calls choose_fusion(...) on calibration
#   5. Saves fold_summary.json + fusion_test_probs.npy + test_labels.npy
# ═══════════════════════════════════════════════════════════════

def _load_v11_state():
    if V11_STATE_F.exists():
        return json.loads(V11_STATE_F.read_text(encoding="utf-8"))
    return {"completed_folds": [], "fold_results": {}}


def _save_v11_state(state):
    V11_STATE_F.write_text(json.dumps(state, indent=2), encoding="utf-8")


def _ingest_existing_fold_summary(state, fold_name):
    """
    If fold_summary.json exists from a prior run, fold it into the state
    file so we skip recomputation. Returns the fold_summary dict (or None).
    """
    fs_path = OUTPUT_ROOT / fold_name / "fold_summary.json"
    if not fs_path.exists():
        return None
    fs = json.loads(fs_path.read_text(encoding="utf-8"))
    if fold_name not in state["completed_folds"]:
        state["completed_folds"].append(fold_name)
        state["fold_results"][fold_name] = fs
        _save_v11_state(state)
    return fs


def run_fold(assets, fold_number, batch_size, v9_tta, seed):
    fold    = make_fold_indices(assets, fold_number)
    out_dir = OUTPUT_ROOT / fold.fold_name
    out_dir.mkdir(parents=True, exist_ok=True)

    log("\n" + "=" * 78)
    log(f"{fold.fold_name.upper()} | train beats={len(fold.train_idx):,} | "
        f"calib beats={len(fold.calib_idx):,} | test beats={len(fold.test_idx):,}")
    log(f"{fold.fold_name.upper()} | train recs={len(fold.train_recs):,} | "
        f"calib recs={len(fold.calib_recs):,} | test recs={len(fold.test_recs):,}")

    # 1. XGBoost expert
    xgb_path = out_dir / f"{fold.fold_name}_xgb.pkl"
    if xgb_path.exists():
        log(f"{fold.fold_name}: loading cached XGBoost expert from disk")
        xgb = joblib.load(xgb_path)
    else:
        log(f"{fold.fold_name}: training XGBoost expert...")
        xgb = train_xgb_expert(fold, assets, out_dir, seed)
    xgb_calib = xgb.predict_proba(assets.features[fold.calib_idx]).astype(np.float32)
    xgb_test  = xgb.predict_proba(assets.features[fold.test_idx]).astype(np.float32)

    # 2. V8 inference
    log(f"{fold.fold_name}: loading V8 checkpoints...")
    v8_models = load_v8_models(fold.fold_name)
    calib_loader = make_loader(fold.calib_idx, assets, batch_size=batch_size, augment=False)
    test_loader  = make_loader(fold.test_idx,  assets, batch_size=batch_size, augment=False)
    t0 = time.time()
    v8_calib = predict_probabilities(v8_models, calib_loader)
    v8_test  = predict_probabilities(v8_models, test_loader)
    log(f"{fold.fold_name}: V8 inference {time.time()-t0:.1f}s")

    # 3. V9 inference with TTA
    log(f"{fold.fold_name}: loading V9 checkpoints...")
    v9_models = load_v9_models(fold.fold_name)
    t0 = time.time()
    v9_calib = predict_probabilities_v9_tta(v9_models, fold.calib_idx, assets,
                                            batch_size=batch_size, n_tta=v9_tta)
    v9_test  = predict_probabilities_v9_tta(v9_models, fold.test_idx, assets,
                                            batch_size=batch_size, n_tta=v9_tta)
    log(f"{fold.fold_name}: V9 TTA inference {time.time()-t0:.1f}s")

    calib_labels = assets.labels[fold.calib_idx]
    test_labels  = assets.labels[fold.test_idx]

    calib_probs = {"v8": v8_calib, "v9": v9_calib, "xgb": xgb_calib}
    test_probs  = {"v8": v8_test,  "v9": v9_test,  "xgb": xgb_test}
    chosen_name, chosen_params, fused_test_probs, calib_best_f1 = choose_fusion(
        calib_labels, calib_probs, test_probs)

    fold_summary = {
        "fold": fold.fold_name,
        "calibration_choice": chosen_name,
        "calibration_f1":     calib_best_f1,
        "choice_params":      chosen_params,
        "experts": {
            "v8":     asdict(evaluate_predictions(test_labels, v8_test,     "v8")),
            "v9":     asdict(evaluate_predictions(test_labels, v9_test,     "v9")),
            "xgb":    asdict(evaluate_predictions(test_labels, xgb_test,    "xgb")),
            "fusion": asdict(evaluate_predictions(test_labels, fused_test_probs, chosen_name)),
        },
    }

    (out_dir / "fold_summary.json").write_text(
        json.dumps(fold_summary, indent=2), encoding="utf-8")
    np.save(out_dir / "fusion_test_probs.npy", fused_test_probs)
    np.save(out_dir / "test_labels.npy", test_labels)
    np.save(out_dir / "v8_test_probs.npy",  v8_test)
    np.save(out_dir / "v9_test_probs.npy",  v9_test)
    np.save(out_dir / "xgb_test_probs.npy", xgb_test)

    log(f"{fold.fold_name}: fusion={chosen_name} | "
        f"F1={fold_summary['experts']['fusion']['f1_macro']:.4f} | "
        f"Acc={fold_summary['experts']['fusion']['accuracy']:.4f} | "
        f"AUROC={fold_summary['experts']['fusion']['auroc']:.4f}")

    # free GPU memory
    del v8_models, v9_models
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return fold_summary


def summarise_runs(fold_summaries):
    fusion_metrics = [item["experts"]["fusion"] for item in fold_summaries]
    v9_metrics     = [item["experts"]["v9"]     for item in fold_summaries]
    v8_metrics     = [item["experts"]["v8"]     for item in fold_summaries]
    xgb_metrics    = [item["experts"]["xgb"]    for item in fold_summaries]

    def block(name, rows):
        return {
            "name":       name,
            "f1_mean":    float(np.mean([r["f1_macro"] for r in rows])),
            "f1_std":     float(np.std( [r["f1_macro"] for r in rows])),
            "acc_mean":   float(np.mean([r["accuracy"] for r in rows])),
            "auroc_mean": float(np.mean([r["auroc"]    for r in rows])),
        }

    return {
        "num_folds":  len(fold_summaries),
        "v11_fusion": block("v11_fusion", fusion_metrics),
        "v9_expert":  block("v9_expert",  v9_metrics),
        "v8_expert":  block("v8_expert",  v8_metrics),
        "xgb_expert": block("xgb_expert", xgb_metrics),
        "folds":      fold_summaries,
    }


def parse_folds(text):
    out = []
    for part in text.split(","):
        v = int(part.strip())
        if v < 1 or v > 5:
            raise ValueError("Fold numbers must be between 1 and 5.")
        out.append(v)
    return sorted(set(out))


log("✅  Cell 8 ready: run_fold (resumable), summarise_runs, _load_v11_state, _ingest_existing_fold_summary")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 9 — RUN EXPERIMENT
#
# Executes (or skips, if cached) all 5 folds and aggregates the metrics.
# Prints a final per-model F1/Acc/AUROC summary inline.
#
# Resumability:
#   - Already-completed folds (per v11_state.json OR a fold_summary.json on
#     disk) are loaded, not recomputed. The console says "⏩ fold_K — skip".
#   - To force a fresh run, delete OUTPUT_ROOT/v11_state.json AND the
#     corresponding fold_K/fold_summary.json files.
# ═══════════════════════════════════════════════════════════════

def run_experiment(folds_text="1,2,3,4,5", batch_size=512, v9_tta=5, seed=42):
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    set_seed(seed)
    log(f"Device: {DEVICE}")
    log("Loading assets...")
    assets = load_assets()
    folds  = parse_folds(folds_text)
    log(f"Running folds: {folds}")

    state = _load_v11_state()
    fold_summaries = []
    for fn in folds:
        fold_name = f"fold_{fn}"

        # 1. Already in state file → skip + load
        if fold_name in state["completed_folds"]:
            fs = state["fold_results"][fold_name]
            log(f"⏩  {fold_name} cached in v11_state.json — skip "
                f"(F1={fs['experts']['fusion']['f1_macro']:.4f})")
            fold_summaries.append(fs)
            continue

        # 2. fold_summary.json on disk but not yet in state file → ingest
        fs = _ingest_existing_fold_summary(state, fold_name)
        if fs is not None:
            log(f"⏩  {fold_name} fold_summary.json found on disk — ingested "
                f"(F1={fs['experts']['fusion']['f1_macro']:.4f})")
            fold_summaries.append(fs)
            continue

        # 3. Otherwise run it
        fs = run_fold(
            assets       = assets,
            fold_number  = fn,
            batch_size   = batch_size,
            v9_tta       = v9_tta,
            seed         = seed + fn,
        )
        state["completed_folds"].append(fold_name)
        state["fold_results"][fold_name] = fs
        _save_v11_state(state)
        fold_summaries.append(fs)

    summary = summarise_runs(fold_summaries)
    (OUTPUT_ROOT / "summary.json").write_text(
        json.dumps(summary, indent=2), encoding="utf-8")

    rows = []
    for k in ("v11_fusion", "v9_expert", "v8_expert", "xgb_expert"):
        b = summary[k]
        rows.append({
            "model":      b["name"],
            "f1_mean":    b["f1_mean"],
            "f1_std":     b["f1_std"],
            "acc_mean":   b["acc_mean"],
            "auroc_mean": b["auroc_mean"],
        })
    df = pd.DataFrame(rows)
    df.to_csv(OUTPUT_ROOT / "summary.csv", index=False)

    # Per-fold CSV (convenient for plots)
    per_fold_rows = []
    for fs in fold_summaries:
        per_fold_rows.append({
            "fold":   fs["fold"],
            "f1_macro": fs["experts"]["fusion"]["f1_macro"],
            "accuracy": fs["experts"]["fusion"]["accuracy"],
            "auroc":    fs["experts"]["fusion"]["auroc"],
        })
    pd.DataFrame(per_fold_rows).to_csv(OUTPUT_ROOT / "summary_all5.csv", index=False)

    print("\n" + "=" * 78)
    print("  V11 5-FOLD RESULTS — patient-aware tri-expert fusion")
    print("=" * 78)
    print(df.to_string(index=False))
    print("=" * 78)
    print(f"  Δ Macro F1 vs V9: "
          f"{summary['v11_fusion']['f1_mean'] - summary['v9_expert']['f1_mean']:+.4f}")
    print(f"  Δ Accuracy vs V9: "
          f"{summary['v11_fusion']['acc_mean'] - summary['v9_expert']['acc_mean']:+.4f}")
    print(f"  Δ AUROC vs V9   : "
          f"{summary['v11_fusion']['auroc_mean'] - summary['v9_expert']['auroc_mean']:+.4f}")
    print("=" * 78)
    return summary, df


# Run all five folds. Already-completed folds are loaded from disk.
summary, df_summary = run_experiment("1,2,3,4,5", batch_size=512, v9_tta=5, seed=42)
df_summary


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 10 — RESULTS PLOTS
#
# Required plots (all generated and saved to OUTPUT_ROOT/plots):
#   1. Per-expert metrics bar chart  (V8 vs V9 vs XGBoost vs V11 fusion)
#   2. Fusion OOF confusion matrix   (raw + normalised, 1×2 subplot)
#   3. ROC curves                    (for all 4 models, all 3 classes)
#   4. Fold-by-fold F1 progression   (per expert, line plot)
#
# Plus extra analytical plots useful for a paper:
#   5. Per-class F1 bars             (V11 fusion vs V9 vs V8 vs XGB)
#   6. Per-fold accuracy + AUROC     (full per-fold metric panel)
#   7. F1 mean ± std error bars      (highlighting the V11 gain)
#
# Everything is computed from the in-memory `summary` + per-fold .npy files
# saved by Cell 9.
# ═══════════════════════════════════════════════════════════════

# Reload per-fold probability arrays so we can build OOF + ROC + per-class F1
fold_names = [fs["fold"] for fs in summary["folds"]]

all_y_true, all_fusion, all_v8, all_v9, all_xgb = [], [], [], [], []
for fn in fold_names:
    d = OUTPUT_ROOT / fn
    all_y_true.append(np.load(d / "test_labels.npy"))
    all_fusion.append(np.load(d / "fusion_test_probs.npy"))
    all_v8.append(np.load(d / "v8_test_probs.npy"))
    all_v9.append(np.load(d / "v9_test_probs.npy"))
    all_xgb.append(np.load(d / "xgb_test_probs.npy"))

y_true_oof = np.concatenate(all_y_true)
fusion_oof = np.concatenate(all_fusion)
v8_oof     = np.concatenate(all_v8)
v9_oof     = np.concatenate(all_v9)
xgb_oof    = np.concatenate(all_xgb)
log(f"OOF predictions assembled: {len(y_true_oof):,} beats")

# ── PLOT 1 — Per-expert metric bar chart ────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
order = ["v11_fusion", "v9_expert", "v8_expert", "xgb_expert"]
labels_pretty = ["V11 fusion", "V9 expert", "V8 expert", "XGBoost expert"]
xs = np.arange(len(order))
w  = 0.27
f1_vals  = [summary[k]["f1_mean"]    for k in order]
acc_vals = [summary[k]["acc_mean"]   for k in order]
au_vals  = [summary[k]["auroc_mean"] for k in order]
b1 = ax.bar(xs - w, f1_vals,  w, color="#3498db", label="Macro F1")
b2 = ax.bar(xs,     acc_vals, w, color="#2ecc71", label="Accuracy")
b3 = ax.bar(xs + w, au_vals,  w, color="#e67e22", label="AUROC")
for bars, vals in [(b1, f1_vals), (b2, acc_vals), (b3, au_vals)]:
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, v + 0.005,
                f"{v:.3f}", ha="center", fontsize=8, fontweight="bold")
ax.set_xticks(xs); ax.set_xticklabels(labels_pretty)
ax.set_ylabel("Score"); ax.set_ylim(0, 1.05)
ax.set_title("V11 vs V9 / V8 / XGBoost — Mean 5-fold Metrics",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "v11_metric_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
log("  Saved: v11_metric_comparison.png")


# ── PLOT 2 — Fusion OOF Confusion Matrix (raw + normalised) ─────────────────
fusion_preds = fusion_oof.argmax(axis=1)
cm  = confusion_matrix(y_true_oof, fusion_preds)
cmn = cm.astype(float) / cm.sum(1, keepdims=True)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
SHORT = ["Normal", "Structural", "Arrhythmia"]
sns.heatmap(cm,  annot=True, fmt="d",   cmap="Greens",
            xticklabels=SHORT, yticklabels=SHORT, ax=axes[0])
axes[0].set_title("OOF Confusion Matrix")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")
sns.heatmap(cmn, annot=True, fmt=".2f", cmap="Greens",
            xticklabels=SHORT, yticklabels=SHORT, ax=axes[1], vmin=0, vmax=1)
axes[1].set_title("OOF Normalized Confusion Matrix")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("True")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "v11_oof_confusion.png", dpi=150, bbox_inches="tight")
plt.show()
log("  Saved: v11_oof_confusion.png")


# ── PLOT 3 — ROC curves for all 4 models ─────────────────────────────────────
y_bin = label_binarize(y_true_oof, classes=[0, 1, 2])
colors_cls = ["#1f77b4", "#ff7f0e", "#2ca02c"]
model_probs = {
    "V11 Fusion": fusion_oof,
    "V9 expert":  v9_oof,
    "V8 expert":  v8_oof,
    "XGBoost":    xgb_oof,
}
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
for ax, (mname, mprobs) in zip(axes.flat, model_probs.items()):
    for ci, (cn, col) in enumerate(zip(SHORT, colors_cls)):
        fpr, tpr, _ = roc_curve(y_bin[:, ci], mprobs[:, ci])
        ax.plot(fpr, tpr, color=col, lw=2, label=f"{cn} (AUC={auc(fpr, tpr):.3f})")
    ax.plot([0, 1], [0, 1], "k--", alpha=0.5)
    ax.set_title(f"OOF ROC — {mname}", fontweight="bold")
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.legend(loc="lower right", fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "v11_oof_roc.png", dpi=150, bbox_inches="tight")
plt.show()
log("  Saved: v11_oof_roc.png")


# ── PLOT 4 — Fold-by-fold F1 progression for every expert ────────────────────
per_fold = {"V11 fusion": [], "V9 expert": [], "V8 expert": [], "XGBoost": []}
for fs in summary["folds"]:
    per_fold["V11 fusion"].append(fs["experts"]["fusion"]["f1_macro"])
    per_fold["V9 expert" ].append(fs["experts"]["v9"]["f1_macro"])
    per_fold["V8 expert" ].append(fs["experts"]["v8"]["f1_macro"])
    per_fold["XGBoost"   ].append(fs["experts"]["xgb"]["f1_macro"])
fig, ax = plt.subplots(figsize=(10, 5.5))
xs = np.arange(1, len(fold_names) + 1)
markers = {"V11 fusion": "o", "V9 expert": "s", "V8 expert": "^", "XGBoost": "D"}
colors  = {"V11 fusion": "#9b59b6", "V9 expert": "#3498db",
           "V8 expert":  "#2ecc71", "XGBoost":   "#e74c3c"}
for k, vs in per_fold.items():
    ax.plot(xs, vs, marker=markers[k], color=colors[k], lw=2, label=k, markersize=8)
    for x, v in zip(xs, vs):
        ax.text(x, v + 0.005, f"{v:.3f}", ha="center", fontsize=7, color=colors[k])
ax.set_xticks(xs); ax.set_xticklabels(fold_names)
ax.set_ylabel("Macro F1"); ax.set_title("Fold-by-Fold Macro F1 Progression",
                                         fontsize=12, fontweight="bold")
ax.set_ylim(0.55, 0.92); ax.grid(alpha=0.3); ax.legend(fontsize=9, loc="center right")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "v11_fold_f1_progression.png", dpi=150, bbox_inches="tight")
plt.show()
log("  Saved: v11_fold_f1_progression.png")


# ── PLOT 5 — Per-class F1 (one bar group per class) ──────────────────────────
def per_class_f1(y_true, probs):
    preds = probs.argmax(axis=1)
    return f1_score(y_true, preds, average=None, zero_division=0)

pc_f1 = {
    "V11 fusion": per_class_f1(y_true_oof, fusion_oof),
    "V9 expert":  per_class_f1(y_true_oof, v9_oof),
    "V8 expert":  per_class_f1(y_true_oof, v8_oof),
    "XGBoost":    per_class_f1(y_true_oof, xgb_oof),
}
fig, ax = plt.subplots(figsize=(11, 5))
xs = np.arange(len(SHORT))
w  = 0.20
for i, (mname, vals) in enumerate(pc_f1.items()):
    bars = ax.bar(xs + (i - 1.5) * w, vals, w, label=mname,
                  color=[colors[mname]] * len(vals), alpha=0.95)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, v + 0.005,
                f"{v:.3f}", ha="center", fontsize=7, fontweight="bold")
ax.set_xticks(xs); ax.set_xticklabels(SHORT)
ax.set_ylabel("Per-class F1"); ax.set_ylim(0, 1.0)
ax.set_title("Per-Class F1 Across Models (OOF, 5-fold)", fontsize=12, fontweight="bold")
ax.legend(fontsize=9, ncol=4); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "v11_per_class_f1.png", dpi=150, bbox_inches="tight")
plt.show()
log("  Saved: v11_per_class_f1.png")


# ── PLOT 6 — Per-fold accuracy & AUROC (extra analysis) ──────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for i, metric in enumerate(["accuracy", "auroc"]):
    ax = axes[i]
    for mkey, mlabel in [("fusion", "V11 fusion"),
                         ("v9",     "V9 expert"),
                         ("v8",     "V8 expert"),
                         ("xgb",    "XGBoost")]:
        vals = [fs["experts"][mkey][metric] for fs in summary["folds"]]
        ax.plot(xs, vals, marker=markers[mlabel], color=colors[mlabel],
                lw=2, label=mlabel, markersize=7)
    ax.set_xticks(xs); ax.set_xticklabels(fold_names)
    ax.set_title(f"Per-fold {metric.upper()}", fontweight="bold")
    ax.set_ylabel(metric.capitalize()); ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "v11_perfold_accuracy_auroc.png", dpi=150, bbox_inches="tight")
plt.show()
log("  Saved: v11_perfold_accuracy_auroc.png")


# ── PLOT 7 — F1 mean ± std bar chart with V11 gain highlighted ───────────────
fig, ax = plt.subplots(figsize=(9, 5))
means = [summary[k]["f1_mean"] for k in order]
stds  = [summary[k]["f1_std"]  for k in order]
bars  = ax.bar(labels_pretty, means, yerr=stds, capsize=6,
               color=["#9b59b6", "#3498db", "#2ecc71", "#e74c3c"],
               edgecolor="white", linewidth=2)
for b, m, s in zip(bars, means, stds):
    ax.text(b.get_x() + b.get_width()/2, m + s + 0.005,
            f"{m:.3f}\n±{s:.3f}", ha="center", fontsize=8, fontweight="bold")
ax.set_ylabel("Macro F1"); ax.set_ylim(0, 1.0)
ax.set_title("Macro F1 (mean ± std over 5 folds)", fontsize=12, fontweight="bold")
ax.grid(axis="y", alpha=0.3)
ax.axhline(summary["v9_expert"]["f1_mean"], color="grey", linestyle="--", lw=1, alpha=0.6,
           label=f"V9 baseline = {summary['v9_expert']['f1_mean']:.4f}")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "v11_f1_meanstd.png", dpi=150, bbox_inches="tight")
plt.show()
log("  Saved: v11_f1_meanstd.png")

# Classification report on fusion OOF
print("\n" + "=" * 78)
print("  V11 Fusion — OOF Classification Report")
print("=" * 78)
print(classification_report(y_true_oof, fusion_preds,
                            target_names=CLASS_NAMES, digits=4))


## XAI Section — 5-layer explainability + V11 fusion-level XAI

This section is the V11 analogue of V9 Cell 11 (XAI). It is **self-contained** — every cell reloads what it needs from disk so the section runs even if the kernel was restarted after the model cells.

**Methods covered** (with the layer of the model they explain):

| Layer | Method | Target |
|---|---|---|
| Waveform input | XAI-1 Integrated Gradients | Per-sample saliency on the V9 sub-expert |
| Last conv block | XAI-2 GradCAM | Class-averaged temporal heatmap (V9 morph.s3.c2) |
| RR features (8) | XAI-3 SHAP GradientExplainer | Per-feature Shapley values per class |
| Waveform input | XAI-4 Occlusion sensitivity | Faithfulness validation of XAI-1 |
| Fusion attention | XAI-5 Cross-attention weights | Rhythm → morphology influence per class |
| Fold-level fusion | V11-A Calibration choice | Which fusion rule won per fold |
| Fold-level fusion | V11-B Δ-vs-V9 per fold | Where the +0.7pp gain comes from |
| Fold-level fusion | V11-C Weighted-grid expert weights | Which expert dominated when grid won |

The 3 V11-specific fusion plots (A/B/C) are **generated fresh from the fold_summary.json files** — they are not just loaded PNGs.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 11 — XAI STARTUP (self-contained reload from disk, V9-Cell-11 style)
#
# WHY SELF-CONTAINED:
#   The kernel may be restarted between training and XAI. So we re-import
#   everything we need, re-open beats.dat, reload the V9 best-fold checkpoint,
#   and re-build the stratified XAI sample (≤500 beats per class for speed).
# ═══════════════════════════════════════════════════════════════════════════════

import os, sys, gc, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
import shap

from captum.attr import IntegratedGradients, LayerGradCam

def _log(msg): print(msg, flush=True)

_log("=" * 78)
_log("  XAI: V11 (5-layer model XAI on V9 sub-expert + 3 fusion-level plots)")
_log("=" * 78)

# Re-anchor paths in case the kernel was restarted between Cell 9 and Cell 11
ROOT             = Path(r"C:\Users\Admin\Desktop\Projects\8th Sem\Codex")
PROJECT_ROOT     = ROOT.parent
BEATS_ROOT       = PROJECT_ROOT / "Preprocessed_Dataset" / "v4_beats"
CHECKPOINT_ROOT  = BEATS_ROOT / "checkpoints"
OUTPUT_ROOT      = ROOT / "outputs" / "v11_triexpert_fusion"
PLOTS_DIR        = OUTPUT_ROOT / "plots"
V9_DIR_CKPT      = CHECKPOINT_ROOT / "v9_wider"
META_FILE        = BEATS_ROOT / "beats_meta.npz"
BEATS_FILE       = BEATS_ROOT / "beats.dat"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES      = ["Normal/Non-Cardiac",
                    "Structural Heart Disease",
                    "Arrhythmia & Electrical"]
N_CLS, N_LEADS, N_RR, BEAT_LEN, SAMPLING_RATE = 3, 12, 8, 300, 500
BEAT_PRE         = int(200 * SAMPLING_RATE / 1000)
DEVICE           = "cuda" if torch.cuda.is_available() else "cpu"
_C3              = ["#2ecc71", "#3498db", "#e74c3c"]

# ── Make sure ResConvBlock/RhythmEncoder/CrossAttentionFusion/DualBranchV9
#    exist in this namespace. If the user is running the whole notebook
#    top-to-bottom they already do (defined in Cell 3). If not, we
#    re-import them by re-declaring small stand-ins. ──────────────────────
if "DualBranchV9" not in dir():
    import math
    class ResConvBlock(nn.Module):
        def __init__(self, ic, oc, k=7, s=1):
            super().__init__()
            p = k // 2
            self.c1 = nn.Conv1d(ic, oc, k, stride=s, padding=p, bias=False)
            self.bn1= nn.BatchNorm1d(oc)
            self.c2 = nn.Conv1d(oc, oc, k, padding=p, bias=False)
            self.bn2= nn.BatchNorm1d(oc)
            self.drop=nn.Dropout(0.10)
            self.skip = (nn.Sequential(nn.Conv1d(ic, oc, 1, stride=s, bias=False),
                                       nn.BatchNorm1d(oc))
                         if (ic != oc or s != 1) else nn.Identity())
        def forward(self, x):
            o = F.gelu(self.bn1(self.c1(x)))
            o = self.drop(self.bn2(self.c2(o)))
            return F.gelu(o + self.skip(x))
    class RhythmEncoder(nn.Module):
        def __init__(self, d=64):
            super().__init__()
            self.fc1=nn.Linear(N_RR,d); self.bn1=nn.BatchNorm1d(d)
            self.fc2=nn.Linear(d,d);    self.bn2=nn.BatchNorm1d(d)
            self.skip=nn.Linear(N_RR,d)
        def forward(self, rr):
            o = F.gelu(self.bn1(self.fc1(rr)))
            o = self.bn2(self.fc2(o))
            return F.gelu(o + self.skip(rr))
    class CrossAttentionFusion(nn.Module):
        def __init__(self, m, r=64, h=2):
            super().__init__()
            d = m // h
            self.heads, self.d_head = h, d
            self.Q=nn.Linear(m,m); self.K=nn.Linear(r,m); self.V=nn.Linear(r,m)
            self.proj=nn.Linear(d,m); self.norm=nn.LayerNorm(m)
            self.scale = math.sqrt(d); self._attn=None
        def forward(self, m, r):
            B = m.size(0)
            q = self.Q(m).view(B, self.heads, self.d_head)
            k = self.K(r).view(B, self.heads, self.d_head)
            v = self.V(r).view(B, self.heads, self.d_head)
            w = F.softmax((q*k).sum(-1)/self.scale, dim=1)
            self._attn = w.detach()
            o = (w.unsqueeze(-1)*v).sum(1)
            return self.norm(m + self.proj(o))
    class MorphologyEncoderV9(nn.Module):
        def __init__(self):
            super().__init__()
            self.stem = nn.Sequential(
                nn.Conv1d(N_LEADS, 96, 9, padding=4, bias=False),
                nn.BatchNorm1d(96), nn.GELU())
            self.s1 = ResConvBlock(96, 96, 7, 2)
            self.s2 = ResConvBlock(96, 192, 5, 2)
            self.s3 = ResConvBlock(192, 384, 5, 1)
            self.gap= nn.AdaptiveAvgPool1d(1)
        def forward(self, x):
            return self.gap(self.s3(self.s2(self.s1(self.stem(x))))).squeeze(-1)
    class DualBranchV9(nn.Module):
        def __init__(self):
            super().__init__()
            self.morph=MorphologyEncoderV9(); self.rhythm=RhythmEncoder(64)
            self.fusion=CrossAttentionFusion(384,64,2)
            self.head=nn.Sequential(
                nn.Linear(384+64,384), nn.GELU(), nn.Dropout(0.4),
                nn.Linear(384,128),    nn.GELU(), nn.Dropout(0.2),
                nn.Linear(128,N_CLS))
        def forward(self, beat, rr):
            m=self.morph(beat); r=self.rhythm(rr)
            return self.head(torch.cat([self.fusion(m,r), r], dim=1))

# ── Load beats + labels + RR features ───────────────────────────────────────
_meta        = np.load(META_FILE, allow_pickle=True)
labels_all   = _meta["labels"].astype(np.int32)
rec_ids_all  = _meta["rec_ids"]
rr_feats_all = _meta["rr_feats"].astype(np.float32)
beats_all    = np.memmap(BEATS_FILE, dtype="float32", mode="r",
                          shape=(len(labels_all), BEAT_LEN, N_LEADS))
_rr_sc       = StandardScaler()
rr_norm_all  = _rr_sc.fit_transform(rr_feats_all).astype(np.float32)
rr_norm_all  = np.nan_to_num(rr_norm_all, nan=0.0, posinf=0.0, neginf=0.0)
_log(f"  Loaded {len(labels_all):,} beats from disk")

# ── Pick the best fold (by V11 fusion F1) for XAI ────────────────────────────
_summary_p = OUTPUT_ROOT / "summary.json"
if not _summary_p.exists():
    raise FileNotFoundError(f"{_summary_p} not found. Run Cell 9 first.")
_summary = json.loads(_summary_p.read_text(encoding="utf-8"))
_fold_table = {fs["fold"]: fs for fs in _summary["folds"]}
_bfn = max(_fold_table, key=lambda k: _fold_table[k]["experts"]["fusion"]["f1_macro"])
_bfi = int(_bfn.split("_")[1]) - 1
_log(f"  Best V11 fold: {_bfn} "
     f"(F1={_fold_table[_bfn]['experts']['fusion']['f1_macro']:.4f}, "
     f"choice={_fold_table[_bfn]['calibration_choice']})")

# ── Load the latest V9 checkpoint from that fold ─────────────────────────────
_xai_model = DualBranchV9().to(DEVICE)
_v9_ckpts  = sorted(V9_DIR_CKPT.glob(f"v9_{_bfn}_ep*.pt"),
                    key=lambda p: int(p.stem.split("_ep")[-1]))
if not _v9_ckpts:
    raise FileNotFoundError(f"No V9 checkpoints found for {_bfn} in {V9_DIR_CKPT}")
_xai_model.load_state_dict(torch.load(_v9_ckpts[-1], map_location=DEVICE))
_xai_model.eval()
_log(f"  Loaded V9 checkpoint: {_v9_ckpts[-1].name}")

# ── Rebuild the test split for the best fold ─────────────────────────────────
from sklearn.model_selection import StratifiedShuffleSplit
unique_recs = np.unique(rec_ids_all)
rec_labels  = np.array([
    int(np.argmax(np.bincount(labels_all[rec_ids_all == r], minlength=N_CLS)))
    for r in unique_recs
], dtype=np.int32)
_sss        = StratifiedShuffleSplit(5, test_size=0.20, random_state=42)
_rec_splits = list(_sss.split(unique_recs, rec_labels))
_te_recs    = set(unique_recs[_rec_splits[_bfi][1]])
_te_b_xai   = np.where(np.array([r in _te_recs for r in rec_ids_all]))[0]
_log(f"  Test set: {len(_te_b_xai):,} beats")

# ── Stratified XAI sample (≤500 beats per class) ─────────────────────────────
_XAI_N_PER_CLASS = 500
np.random.seed(42)
_xai_idx = []
for _ci in range(3):
    _cls = _te_b_xai[labels_all[_te_b_xai] == _ci]
    _pick= np.random.choice(_cls, min(_XAI_N_PER_CLASS, len(_cls)), replace=False)
    _xai_idx.extend(_pick.tolist())
_xai_idx = np.array(_xai_idx)
_log(f"  XAI sample size: {len(_xai_idx)} beats (~500/class)")

_xai_beats  = torch.tensor(beats_all[_xai_idx].transpose(0, 2, 1), dtype=torch.float32)
_xai_rr     = torch.tensor(rr_norm_all[_xai_idx], dtype=torch.float32)
_xai_labels = labels_all[_xai_idx]

def _cls_idx_local(c):
    return np.where(_xai_labels == c)[0]

_t_axis      = np.arange(BEAT_LEN) / SAMPLING_RATE * 1000
_BEAT_PRE_MS = BEAT_PRE / SAMPLING_RATE * 1000


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 12 — XAI-1: INTEGRATED GRADIENTS
#
# IG is the primary attribution method for the waveform branch. It is faithful
# (completeness axiom) and gives a per-sample, per-lead attribution value.
# We freeze RR at its actual value per sample and compute IG w.r.t. the beat.
#
# Output plots:
#   - xai1_integrated_gradients_per_class.png : per-class mean |IG| over the beat
#   - xai1_ig_lead_heatmap.png                : per-class 12-lead × 300-sample heatmap
# ═══════════════════════════════════════════════════════════════════════════════

_log("\n[1/6] XAI-1: Integrated Gradients (beat waveform attribution)...")

class _IGWrapper(nn.Module):
    def __init__(self, model, rr_batch):
        super().__init__()
        self.model = model
        self.rr_batch = rr_batch
    def forward(self, beat):
        bsz = beat.size(0)
        if bsz > self.rr_batch.size(0):
            n_rep = bsz // self.rr_batch.size(0)
            rr_e  = self.rr_batch.repeat_interleave(n_rep, dim=0)
            if rr_e.size(0) != bsz:
                rr_e = self.rr_batch.repeat(n_rep + 1, 1)[:bsz]
            return self.model(beat, rr_e)
        return self.model(beat, self.rr_batch)

_len_xai  = len(_xai_idx)
_ig_attrs = np.zeros((_len_xai, N_LEADS, BEAT_LEN), dtype=np.float32)
_IG_BATCH = 32

for _start in range(0, _len_xai, _IG_BATCH):
    _end   = min(_start + _IG_BATCH, _len_xai)
    _bb    = _xai_beats[_start:_end].to(DEVICE)
    _br    = _xai_rr[_start:_end].to(DEVICE)
    _bl    = _xai_labels[_start:_end]
    _wrap  = _IGWrapper(_xai_model, _br)
    _ig    = IntegratedGradients(_wrap)
    _base  = torch.zeros_like(_bb)
    for _tc in range(3):
        _mask = np.where(_bl == _tc)[0]
        if len(_mask) == 0:
            continue
        _attr = _ig.attribute(_bb[_mask], baselines=_base[_mask],
                              target=_tc, n_steps=50)
        _ig_attrs[_start + _mask] = _attr.detach().cpu().numpy()

_log(f"  IG attribution shape: {_ig_attrs.shape}")

# Plot 1A — per-class mean |IG| over the beat
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("XAI-1: Integrated Gradients — Mean |IG| Over Beat (averaged across 12 leads)",
             fontsize=12, fontweight="bold")
for _ci, (cn, col) in enumerate(zip(CLASS_NAMES, _C3)):
    _m = _cls_idx_local(_ci)
    _mean = np.abs(_ig_attrs[_m]).mean(axis=(0, 1))
    _sem  = np.abs(_ig_attrs[_m]).mean(axis=1).std(axis=0) / np.sqrt(len(_m))
    ax = axes[_ci]
    ax.fill_between(_t_axis, _mean - _sem, _mean + _sem, alpha=0.3, color=col)
    ax.plot(_t_axis, _mean, color=col, lw=2)
    ax.axvline(_BEAT_PRE_MS, color="black", linestyle="--", lw=1.5, alpha=0.6, label="R-peak")
    ax.axvspan(0,                _BEAT_PRE_MS*0.6,  alpha=0.05, color="blue",  label="P-wave")
    ax.axvspan(_BEAT_PRE_MS*0.85,_BEAT_PRE_MS*1.15, alpha=0.05, color="red",   label="QRS")
    ax.axvspan(_BEAT_PRE_MS*1.3, _BEAT_PRE_MS*2.0,  alpha=0.05, color="green", label="ST/T-wave")
    ax.set_title(cn[:22], fontsize=9, fontweight="bold", color=col)
    ax.set_xlabel("Time (ms)"); ax.set_ylabel("|IG attribution|")
    ax.legend(fontsize=6); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "xai1_integrated_gradients_per_class.png", dpi=150, bbox_inches="tight")
plt.show()
_log("  Saved: xai1_integrated_gradients_per_class.png")

# Plot 1B — 12-lead × 300-sample heatmap per class
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("XAI-1: Integrated Gradients — Per-Lead Heatmap (12 leads × 300 samples)",
             fontsize=11, fontweight="bold")
LEAD_LBL = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]
for _ci, (cn, ax) in enumerate(zip(CLASS_NAMES, axes)):
    _m = _cls_idx_local(_ci)
    _hm = np.abs(_ig_attrs[_m]).mean(axis=0)
    im = ax.imshow(_hm, aspect="auto", cmap="hot", origin="upper",
                   extent=[0, BEAT_LEN/SAMPLING_RATE*1000, 11.5, -0.5])
    ax.set_yticks(range(12)); ax.set_yticklabels(LEAD_LBL, fontsize=8)
    ax.axvline(_BEAT_PRE_MS, color="cyan", lw=1.5, linestyle="--", label="R-peak")
    ax.set_xlabel("Time (ms)"); ax.set_title(cn[:22], fontsize=9)
    plt.colorbar(im, ax=ax, fraction=0.04, label="|IG|")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "xai1_ig_lead_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
_log("  Saved: xai1_ig_lead_heatmap.png")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 13 — XAI-2: GradCAM + class-averaged heatmaps + IG agreement
#
# GradCAM is applied to the final convolutional layer of the V9 morphology
# encoder (`morph.s3.c2`). It gives a coarse temporal map of where the CNN's
# final feature map is most discriminative per class. We then overlay IG (XAI-1)
# on top to test agreement (Pearson r). High agreement = both methods point at
# the same regions, which strengthens the explanation.
# ═══════════════════════════════════════════════════════════════════════════════

_log("\n[2/6] XAI-2: GradCAM on V9.morph.s3.c2 + IG agreement check...")

_target_layer = _xai_model.morph.s3.c2

class _GradCamWrapper(nn.Module):
    def __init__(self, model, rr_batch):
        super().__init__()
        self.model = model
        self.rr_batch = rr_batch
    def forward(self, beat):
        bsz = beat.size(0); orig = self.rr_batch.size(0)
        if bsz != orig:
            if bsz < orig:
                rr_in = self.rr_batch[:bsz]
            else:
                nr = (bsz // orig) + 1
                rr_in = self.rr_batch.repeat(nr, 1)[:bsz]
            return self.model(beat, rr_in)
        return self.model(beat, self.rr_batch)

_gradcam_maps = np.zeros((_len_xai, BEAT_LEN), dtype=np.float32)
for _start in range(0, _len_xai, _IG_BATCH):
    _end = min(_start + _IG_BATCH, _len_xai)
    _bb  = _xai_beats[_start:_end].to(DEVICE)
    _br  = _xai_rr[_start:_end].to(DEVICE)
    _bl  = _xai_labels[_start:_end]
    _gw  = _GradCamWrapper(_xai_model, _br)
    _lgc = LayerGradCam(_gw, _target_layer)
    for _tc in range(3):
        _mask = np.where(_bl == _tc)[0]
        if len(_mask) == 0:
            continue
        _attr = _lgc.attribute(_bb[_mask], target=_tc)
        _mean = _attr.mean(dim=1, keepdim=True)
        _up   = F.interpolate(_mean, size=BEAT_LEN, mode="linear", align_corners=False)
        _up   = _up.squeeze(1).detach().cpu().numpy()
        _gradcam_maps[_start + _mask] = np.maximum(_up, 0)

# Plot 2A — class-averaged GradCAM curves
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("XAI-2: GradCAM (V9 morph.s3.c2) — Class-Averaged Heatmaps",
             fontsize=12, fontweight="bold")
for _ci, (cn, col, ax) in enumerate(zip(CLASS_NAMES, _C3, axes)):
    _m = _cls_idx_local(_ci)
    _gc = _gradcam_maps[_m].mean(axis=0)
    _n  = (_gc - _gc.min()) / (_gc.max() - _gc.min() + 1e-8)
    ax.fill_between(_t_axis, _n, alpha=0.35, color=col)
    ax.plot(_t_axis, _n, color=col, lw=2)
    ax.axvline(_BEAT_PRE_MS, color="black", linestyle="--", lw=1.5, label="R-peak")
    ax.set_title(cn[:22], fontsize=9, fontweight="bold", color=col)
    ax.set_xlabel("Time (ms)"); ax.set_ylabel("Norm. GradCAM")
    ax.legend(fontsize=7); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "xai2_gradcam_per_class.png", dpi=150, bbox_inches="tight")
plt.show()
_log("  Saved: xai2_gradcam_per_class.png")

# Plot 2B — GradCAM vs IG agreement
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("XAI-2 vs XAI-1: GradCAM (solid) vs IG (dashed) Agreement",
             fontsize=11, fontweight="bold")
for _ci, (cn, col, ax) in enumerate(zip(CLASS_NAMES, _C3, axes)):
    _m = _cls_idx_local(_ci)
    _ig = np.abs(_ig_attrs[_m]).mean(axis=(0, 1))
    _ign= (_ig - _ig.min()) / (_ig.max() - _ig.min() + 1e-8)
    _gc = _gradcam_maps[_m].mean(axis=0)
    _gcn= (_gc - _gc.min()) / (_gc.max() - _gc.min() + 1e-8)
    ax.plot(_t_axis, _gcn, color=col, lw=2.5, label="GradCAM")
    ax.plot(_t_axis, _ign, color=col, lw=1.5, linestyle="--", alpha=0.8, label="IG")
    ax.axvline(_BEAT_PRE_MS, color="black", linestyle=":", lw=1.5)
    _corr = np.corrcoef(_ign, _gcn)[0, 1]
    ax.set_title(f"{cn[:20]}\n(r={_corr:.3f})", fontsize=8, fontweight="bold")
    ax.set_xlabel("Time (ms)"); ax.legend(fontsize=7); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "xai2_gradcam_vs_ig_agreement.png", dpi=150, bbox_inches="tight")
plt.show()
_log("  Saved: xai2_gradcam_vs_ig_agreement.png")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 14 — XAI-3: SHAP GradientExplainer on the RR feature branch
#
# WHY: The CNN reads the raw beat waveform; the RR branch reads 8 HRV features.
# IG/GradCAM cover the waveform; SHAP here gives per-feature Shapley scores
# for the RR side. Averaged over 5 random background batches to stabilise.
# ═══════════════════════════════════════════════════════════════════════════════

_log("\n[3/6] XAI-3: SHAP GradientExplainer on RR feature branch...")

class _RROnlyWrapper(nn.Module):
    def __init__(self, model, beat_batch):
        super().__init__()
        self.model = model; self.beat_batch = beat_batch
    def forward(self, rr):
        return self.model(self.beat_batch, rr)

_SHAP_BG, _SHAP_EVAL = 200, 300
_bg_idx = np.random.choice(len(_xai_idx), min(_SHAP_BG,   len(_xai_idx)), replace=False)
_ev_idx = np.random.choice(len(_xai_idx), min(_SHAP_EVAL, len(_xai_idx)), replace=False)
_bg_beats = _xai_beats[_bg_idx].to(DEVICE)
_bg_rr    = _xai_rr[_bg_idx].to(DEVICE)
_ev_beats = _xai_beats[_ev_idx].to(DEVICE)
_ev_rr    = _xai_rr[_ev_idx].to(DEVICE)
_ev_labels= _xai_labels[_ev_idx]

_shap_vals_all = []
for _bi in range(5):
    _bg_pick = np.random.choice(len(_bg_idx), min(50, len(_bg_idx)), replace=False)
    _wrap    = _RROnlyWrapper(_xai_model, _ev_beats)
    try:
        _exp = shap.GradientExplainer(_wrap, _bg_rr[_bg_pick])
        _sv  = _exp.shap_values(_ev_rr, nsamples=50)
        _shap_vals_all.append(np.stack(_sv, axis=-1))   # (N_ev, N_RR, N_CLS)
    except Exception as e:
        _log(f"  SHAP batch {_bi} failed: {e}")

if _shap_vals_all:
    _shap_vals = np.stack(_shap_vals_all, axis=0).mean(axis=0)
    _log(f"  SHAP values shape: {_shap_vals.shape}")

    _RR_NAMES = ["RR Mean (ms)", "RR Std (ms)", "RR CV", "RR Min", "RR Max",
                 "RMSSD", "Heart Rate", "HR Variability"]

    # 3A — mean |SHAP| per feature per class
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle("XAI-3: SHAP on RR Branch — Per-Class Feature Contributions",
                 fontsize=12, fontweight="bold")
    for _ci, (cn, col, ax) in enumerate(zip(CLASS_NAMES, _C3, axes)):
        _mabs = np.abs(_shap_vals[_ev_labels == _ci, :, _ci]).mean(axis=0)
        _so   = np.argsort(_mabs)
        _bc   = [col if i == _so[-1] else "#bdc3c7" for i in range(N_RR)]
        ax.barh([_RR_NAMES[i] for i in _so], _mabs[_so], color=[_bc[i] for i in _so])
        ax.set_title(cn[:22], fontsize=9, fontweight="bold", color=col)
        ax.set_xlabel("Mean |SHAP value|"); ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "xai3_shap_rr_features.png", dpi=150, bbox_inches="tight")
    plt.show()
    _log("  Saved: xai3_shap_rr_features.png")

    # 3B — beeswarm-style scatter per class
    for _ci, cn in enumerate(CLASS_NAMES):
        _m = np.where(_ev_labels == _ci)[0]
        if len(_m) < 5:
            continue
        _sv_c = _shap_vals[_m, :, _ci]
        _rv_c = _ev_rr[_m].cpu().numpy()
        fig, ax = plt.subplots(figsize=(9, 5))
        for _fi in range(N_RR):
            _jit = np.random.uniform(-0.2, 0.2, len(_m))
            sc = ax.scatter(_sv_c[:, _fi], np.full(len(_m), _fi) + _jit,
                            c=_rv_c[:, _fi], cmap="coolwarm", alpha=0.5, s=8)
        ax.set_yticks(range(N_RR)); ax.set_yticklabels(_RR_NAMES)
        ax.axvline(0, color="black", lw=1)
        ax.set_xlabel("SHAP value → class probability")
        ax.set_title(f"XAI-3: SHAP Beeswarm — {cn}", fontsize=10, fontweight="bold")
        plt.colorbar(sc, ax=ax, label="Normalised feature value")
        ax.grid(alpha=0.2)
        plt.tight_layout()
        _cname = cn.replace("/", "_").replace(" ", "_").replace("&", "and")
        plt.savefig(PLOTS_DIR / f"xai3_shap_beeswarm_{_cname}.png", dpi=150, bbox_inches="tight")
        plt.show()
        _log(f"  Saved: xai3_shap_beeswarm_{_cname}.png")
else:
    _log("  ⚠ All SHAP batches failed; skipping XAI-3 plots.")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 15 — XAI-4: OCCLUSION SENSITIVITY + FAITHFULNESS VALIDATION
#
# Slide a 40 ms zero-mask across the beat; record P(true class) drop.
# Big drops = the model genuinely depends on that window. This empirically
# validates XAI-1 (IG) and XAI-2 (GradCAM) — methods are faithful if their
# top regions match the occlusion peaks.
# ═══════════════════════════════════════════════════════════════════════════════

_log("\n[4/6] XAI-4: Occlusion sensitivity + faithfulness validation...")

_OCC_WIN  = 20
_OCC_STEP = 10
_n_pos    = (BEAT_LEN - _OCC_WIN) // _OCC_STEP + 1
_occ_drop = np.zeros((_len_xai, _n_pos), dtype=np.float32)

_baseline_probs = np.zeros(_len_xai, dtype=np.float32)
with torch.no_grad():
    for _start in range(0, _len_xai, 128):
        _end = min(_start + 128, _len_xai)
        _xb  = _xai_beats[_start:_end].to(DEVICE)
        _rb  = _xai_rr[_start:_end].to(DEVICE)
        _p   = F.softmax(_xai_model(_xb, _rb), dim=1).cpu().numpy()
        for _i, _gi in enumerate(range(_start, _end)):
            _baseline_probs[_gi] = _p[_i, _xai_labels[_gi]]

with torch.no_grad():
    for _pi, _pos in enumerate(range(0, BEAT_LEN - _OCC_WIN + 1, _OCC_STEP)):
        _ob = _xai_beats.clone()
        _ob[:, :, _pos:_pos+_OCC_WIN] = 0.0
        for _start in range(0, _len_xai, 128):
            _end = min(_start + 128, _len_xai)
            _xb  = _ob[_start:_end].to(DEVICE)
            _rb  = _xai_rr[_start:_end].to(DEVICE)
            _p   = F.softmax(_xai_model(_xb, _rb), dim=1).cpu().numpy()
            for _i, _gi in enumerate(range(_start, _end)):
                _occ_drop[_gi, _pi] = float(_baseline_probs[_gi] - _p[_i, _xai_labels[_gi]])

# Upsample to BEAT_LEN
_occ_up = np.zeros((_len_xai, BEAT_LEN), dtype=np.float32)
for _gi in range(_len_xai):
    _t = torch.tensor(_occ_drop[_gi:_gi+1, np.newaxis, :])
    _u = F.interpolate(_t, size=BEAT_LEN, mode="linear", align_corners=False)
    _occ_up[_gi] = _u.squeeze().numpy()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("XAI-4: Occlusion Sensitivity — P(true class) drop when 40 ms masked",
             fontsize=11, fontweight="bold")
for _ci, (cn, col, ax) in enumerate(zip(CLASS_NAMES, _C3, axes)):
    _m  = _cls_idx_local(_ci)
    _md = _occ_up[_m].mean(axis=0)
    _sd = _occ_up[_m].std(axis=0) / np.sqrt(len(_m))
    ax.fill_between(_t_axis, _md - _sd, _md + _sd, alpha=0.25, color=col)
    ax.plot(_t_axis, _md, color=col, lw=2)
    ax.axvline(_BEAT_PRE_MS, color="black", linestyle="--", lw=1.5, label="R-peak")
    ax.axhline(0, color="grey", lw=0.8)
    ax.set_title(cn[:22], fontsize=9, fontweight="bold", color=col)
    ax.set_xlabel("Occluded window start (ms)"); ax.set_ylabel("ΔP")
    ax.legend(fontsize=7); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "xai4_occlusion_sensitivity.png", dpi=150, bbox_inches="tight")
plt.show()
_log("  Saved: xai4_occlusion_sensitivity.png")

# Faithfulness validation: top-20% IG vs top-20% Occlusion overlap
_log("\n  Faithfulness: IG vs Occlusion (top-20% region overlap + Pearson r)")
_faith = []
for _ci, cn in enumerate(CLASS_NAMES):
    _m   = _cls_idx_local(_ci)
    _igm = np.abs(_ig_attrs[_m]).mean(axis=(0, 1))
    _ocm = _occ_up[_m].mean(axis=0)
    _topk = int(BEAT_LEN * 0.20)
    _i_top = set(np.argsort(_igm)[-_topk:])
    _o_top = set(np.argsort(_ocm)[-_topk:])
    _ov    = len(_i_top & _o_top) / _topk
    _r     = float(np.corrcoef(_igm, _ocm)[0, 1])
    _faith.append({"Class": cn,
                   "IG–Occlusion overlap (top 20%)": f"{_ov:.2%}",
                   "Pearson r": f"{_r:.3f}"})
    _log(f"    {cn[:25]}: overlap={_ov:.2%}, r={_r:.3f}")
pd.DataFrame(_faith).to_csv(PLOTS_DIR / "xai_faithfulness_validation.csv", index=False)


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 16 — XAI-5: CROSS-ATTENTION WEIGHTS (rhythm → morphology)
#
# The CrossAttentionFusion block stores its attention map on ._attn after
# every forward pass. We aggregate by class to ask: "how much does the model
# rely on RR features when interpreting morphology?" — expected pattern:
# Arrhythmia > Structural > Normal.
# ═══════════════════════════════════════════════════════════════════════════════

_log("\n[5/6] XAI-5: Cross-attention weights (rhythm → morphology)...")

_attn_by_cls   = {0: [], 1: [], 2: []}
_attn_per_head = {0: {0: [], 1: []}, 1: {0: [], 1: []}, 2: {0: [], 1: []}}
_xai_model.eval()
with torch.no_grad():
    for _start in range(0, _len_xai, 256):
        _end = min(_start + 256, _len_xai)
        _xb  = _xai_beats[_start:_end].to(DEVICE)
        _rb  = _xai_rr[_start:_end].to(DEVICE)
        _    = _xai_model(_xb, _rb)
        _aw  = _xai_model.fusion._attn
        if _aw is not None:
            _an = _aw.cpu().numpy()
            for _bi in range(_an.shape[0]):
                _gi = _start + _bi
                if _gi >= _len_xai: break
                _c = int(_xai_labels[_gi])
                _attn_by_cls[_c].append(float(_an[_bi].mean()))
                for _hi in range(min(2, _an.shape[1])):
                    _attn_per_head[_c][_hi].append(float(_an[_bi, _hi]))

_means = [np.mean(_attn_by_cls[c]) if _attn_by_cls[c] else 0. for c in range(3)]
_stds  = [np.std(_attn_by_cls[c])  if _attn_by_cls[c] else 0. for c in range(3)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("XAI-5: Cross-Attention Weights — How RR Features Modulate Morphology",
             fontsize=12, fontweight="bold")
bars = axes[0].bar(CLASS_NAMES, _means, yerr=_stds, color=_C3,
                   capsize=5, edgecolor="white", linewidth=1.5, alpha=0.85)
axes[0].set_ylabel("Mean Cross-Attention Weight (both heads)")
axes[0].set_title("Mean attention weight ± std")
axes[0].tick_params(axis="x", rotation=15); axes[0].grid(axis="y", alpha=0.3)
for b, m, s in zip(bars, _means, _stds):
    axes[0].text(b.get_x() + b.get_width()/2, m + s + 0.001,
                 f"{m:.4f}", ha="center", fontsize=9, fontweight="bold")

_h0 = [np.mean(_attn_per_head[c][0]) if _attn_per_head[c][0] else 0 for c in range(3)]
_h1 = [np.mean(_attn_per_head[c][1]) if _attn_per_head[c][1] else 0 for c in range(3)]
_x3 = np.arange(3); _w3 = 0.3
axes[1].bar(_x3 - _w3/2, _h0, _w3, color=_C3, alpha=0.9, label="Head 0")
axes[1].bar(_x3 + _w3/2, _h1, _w3, color=_C3, alpha=0.5, label="Head 1", hatch="//")
axes[1].set_xticks(_x3); axes[1].set_xticklabels(CLASS_NAMES, rotation=15, ha="right", fontsize=8)
axes[1].set_ylabel("Mean attention weight")
axes[1].set_title("Per-head weights (do heads specialise?)")
axes[1].legend(fontsize=8); axes[1].grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "xai5_cross_attention_weights.png", dpi=150, bbox_inches="tight")
plt.show()
_log("  Saved: xai5_cross_attention_weights.png")

_log("\n  Clinical interpretation:")
_notes = [
    "Normal     : low RR→morphology attention (regular rhythm → no rhythm-based guidance needed)",
    "Structural : moderate RR attention (structural class is mainly morphological)",
    "Arrhythmia : highest RR attention (rhythm irregularity directly defines the class)",
]
for cn, m, note in zip(CLASS_NAMES, _means, _notes):
    _log(f"    {cn[:28]}: mean_attn={m:.4f} | {note}")


### V11-specific fusion XAI

The three plots below are **generated fresh from the per-fold `fold_summary.json` files**, not loaded from cached PNGs. They explain how the V11 fusion layer itself made decisions — a kind of explainability the underlying V8/V9/XGB experts can't give on their own.

- **A. Calibration choice per fold** (Cell 17): how often each fusion rule (`meta_logreg` vs `weighted_grid`) won on calibration.
- **B. Δ-vs-V9 per fold** (Cell 18): the per-fold breakdown of the +0.7pp F1 gain.
- **C. Weighted-grid expert reliance** (Cell 19): when `weighted_grid` won, which expert carried most of the weight.

Each cell shows the computation step explicitly so the figure is reproducible.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 17 — V11 FUSION XAI A: Calibration strategy chosen per fold
#
# Reads OUTPUT_ROOT/fold_{1..5}/fold_summary.json directly, counts how often
# each calibration strategy won, and plots the distribution. This is the
# fusion-level analogue of "feature importance" — except the 'features' here
# are the candidate fusion rules themselves.
# ═══════════════════════════════════════════════════════════════════════════════

_log("\n[6a/6] V11 Fusion XAI A: Calibration strategy chosen per fold")

_fold_dirs = sorted([p for p in OUTPUT_ROOT.glob("fold_*") if p.is_dir()],
                    key=lambda p: int(p.name.split("_")[1]))

_choices = {}
for _fd in _fold_dirs:
    _fs_p = _fd / "fold_summary.json"
    if not _fs_p.exists():
        continue
    _fs = json.loads(_fs_p.read_text(encoding="utf-8"))
    _choices[_fs["fold"]] = _fs["calibration_choice"]
_log(f"  Per-fold choices: {_choices}")

# Tabulate and plot
import collections
_cnt = collections.Counter(_choices.values())
_keys = sorted(_cnt.keys())
_vals = [_cnt[k] for k in _keys]

fig, ax = plt.subplots(figsize=(8, 5))
_palette = {"meta_logreg": "#4c72b0", "weighted_grid": "#dd8452",
            "v8": "#55a868", "v9": "#c44e52", "xgb": "#8172b3"}
_bcolors = [_palette.get(k, "#7f7f7f") for k in _keys]
bars = ax.bar(_keys, _vals, color=_bcolors, edgecolor="white", linewidth=1.5)
for b, v in zip(bars, _vals):
    ax.text(b.get_x() + b.get_width()/2, v + 0.03,
            f"{v}", ha="center", fontsize=11, fontweight="bold")
ax.set_ylabel("Number of folds")
ax.set_title("V11 Fusion-Level XAI: Calibration Strategy Chosen Per Fold",
             fontsize=12, fontweight="bold")
ax.set_ylim(0, max(_vals) + 1.0)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "v11_xai_choice_counts.png", dpi=150, bbox_inches="tight")
plt.show()
_log("  Saved: v11_xai_choice_counts.png")

# Per-fold breakdown table
_per_fold_choice_df = pd.DataFrame([
    {"fold": k, "calibration_choice": v} for k, v in _choices.items()
])
print(_per_fold_choice_df.to_string(index=False))


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 18 — V11 FUSION XAI B: Δ Macro F1 vs V9 per fold
#
# For each fold, compute (F1 of V11 fusion) - (F1 of V9 expert). Positive
# values show the fusion meaningfully improved over the strongest single
# expert. If the gain were near zero for every fold, the fusion layer would
# be redundant. Showing the bars per fold is more honest than a single mean.
# ═══════════════════════════════════════════════════════════════════════════════

_log("\n[6b/6] V11 Fusion XAI B: Δ Macro F1 vs V9 per fold")

_deltas = []
for _fd in _fold_dirs:
    _fs_p = _fd / "fold_summary.json"
    if not _fs_p.exists():
        continue
    _fs = json.loads(_fs_p.read_text(encoding="utf-8"))
    _deltas.append({
        "fold":           _fs["fold"],
        "v9_f1":          _fs["experts"]["v9"]["f1_macro"],
        "v11_fusion_f1":  _fs["experts"]["fusion"]["f1_macro"],
        "delta":          _fs["experts"]["fusion"]["f1_macro"] - _fs["experts"]["v9"]["f1_macro"],
    })
_df_delta = pd.DataFrame(_deltas)
print(_df_delta.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(_df_delta["fold"], _df_delta["delta"], color="#2ecc71",
              edgecolor="white", linewidth=1.5)
for b, v in zip(bars, _df_delta["delta"]):
    ax.text(b.get_x() + b.get_width()/2, v + 0.0002,
            f"{v:+.4f}", ha="center", fontsize=9, fontweight="bold")
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("Δ Macro F1  (V11 fusion − V9 expert)")
ax.set_title("V11 Fusion-Level XAI: Gain Over V9 by Fold",
             fontsize=12, fontweight="bold")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "v11_xai_delta_vs_v9.png", dpi=150, bbox_inches="tight")
plt.show()
_log("  Saved: v11_xai_delta_vs_v9.png")
_log(f"  Mean Δ F1 = {_df_delta['delta'].mean():+.4f}")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 19 — V11 FUSION XAI C: Weighted-grid expert reliance
#
# When weighted_grid wins, the chosen weights (one per expert) are stored
# under fold_summary["choice_params"]. This plot stacks them per fold to
# show which expert the fusion leaned on hardest. Folds that chose
# meta_logreg are excluded since they don't expose interpretable expert
# weights (the logreg coefficients are over a 15-feature meta-vector).
# ═══════════════════════════════════════════════════════════════════════════════

_log("\n[6c/6] V11 Fusion XAI C: Weighted-grid expert reliance")

_wgrid_rows = []
for _fd in _fold_dirs:
    _fs_p = _fd / "fold_summary.json"
    if not _fs_p.exists():
        continue
    _fs = json.loads(_fs_p.read_text(encoding="utf-8"))
    if _fs["calibration_choice"] != "weighted_grid":
        continue
    cp = _fs["choice_params"]
    _wgrid_rows.append({
        "fold": _fs["fold"],
        "V8 expert":      float(cp.get("v8",  0.0)),
        "V9 expert":      float(cp.get("v9",  0.0)),
        "XGBoost expert": float(cp.get("xgb", 0.0)),
    })

if _wgrid_rows:
    _df_w = pd.DataFrame(_wgrid_rows)
    print(_df_w.to_string(index=False))

    fig, ax = plt.subplots(figsize=(10, 5))
    _xs    = np.arange(len(_df_w))
    _bot   = np.zeros(len(_df_w))
    _exps  = ["V8 expert", "V9 expert", "XGBoost expert"]
    _cols  = ["#4c72b0", "#55a868", "#dd8452"]
    for _e, _c in zip(_exps, _cols):
        _v = _df_w[_e].values
        ax.bar(_xs, _v, bottom=_bot, color=_c, edgecolor="white", linewidth=1.2, label=_e)
        for _xi, (_vv, _bb) in enumerate(zip(_v, _bot)):
            if _vv > 0.02:
                ax.text(_xi, _bb + _vv/2, f"{_vv:.2f}",
                        ha="center", va="center", fontsize=8,
                        color="white", fontweight="bold")
        _bot += _v
    ax.set_xticks(_xs); ax.set_xticklabels(_df_w["fold"])
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Expert weight")
    ax.set_title("V11 Fusion-Level XAI: Weighted-Grid Expert Reliance",
                 fontsize=12, fontweight="bold")
    ax.legend(fontsize=9, loc="upper right")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "v11_xai_weighted_expert_mix.png", dpi=150, bbox_inches="tight")
    plt.show()
    _log("  Saved: v11_xai_weighted_expert_mix.png")
else:
    _log("  No folds chose weighted_grid; expert-mix plot skipped.")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 20 — XAI SUMMARY TABLE
#
# Final table that lists every XAI method used, what it explains, and why we
# included it. Useful for the methods section of the paper. The table is
# also saved to CSV alongside the plots.
# ═══════════════════════════════════════════════════════════════════════════════

_log("\n[Summary] XAI method matrix")

_xai_method_matrix = pd.DataFrame([
    {"Method": "XAI-1 Integrated Gradients",
     "Scope":  "Waveform input (V9)",
     "Output": "Per-sample × per-lead attribution",
     "Why used": "Faithful gradient attribution for differentiable ECG CNN",
     "Best for": "P/QRS/T-wave saliency at the beat level"},
    {"Method": "XAI-2 GradCAM",
     "Scope":  "Last conv layer V9.morph.s3.c2",
     "Output": "Class-averaged temporal heatmap",
     "Why used": "Localises CNN focus in the final feature map",
     "Best for": "QRS-region localisation"},
    {"Method": "XAI-3 SHAP GradientExplainer",
     "Scope":  "8 RR features (rhythm branch)",
     "Output": "Per-class Shapley values per feature",
     "Why used": "Additive attribution for the feature-based branch",
     "Best for": "Identifying dominant HRV feature per disease class"},
    {"Method": "XAI-4 Occlusion Sensitivity",
     "Scope":  "Beat waveform, 40 ms sliding windows",
     "Output": "ΔP(true class) curve",
     "Why used": "Faithfulness validation of XAI-1 and XAI-2",
     "Best for": "Empirical sanity check on attribution"},
    {"Method": "XAI-5 Cross-Attention Weights",
     "Scope":  "CrossAttentionFusion block (V9)",
     "Output": "Per-class attention scalar (+ per-head)",
     "Why used": "Architectural — measures rhythm→morphology influence",
     "Best for": "Rhythm-vs-morphology dominance per disease class"},
    {"Method": "V11-A Calibration Choice",
     "Scope":  "Fold-level fusion selection",
     "Output": "Per-fold winning fusion strategy",
     "Why used": "Explains the V11 calibration-time decision",
     "Best for": "Showing the fusion isn't a single hard-coded rule"},
    {"Method": "V11-B Δ-vs-V9 per fold",
     "Scope":  "Fold-level F1 gain",
     "Output": "Per-fold ΔF1 over V9 expert",
     "Why used": "Verifies the +0.7pp F1 gain is consistent",
     "Best for": "Defending the fusion's headline number"},
    {"Method": "V11-C Weighted-grid Expert Mix",
     "Scope":  "Weighted-grid folds only",
     "Output": "Stacked expert weights per fold",
     "Why used": "Quantifies which expert the fusion leaned on",
     "Best for": "Interpreting the late-fusion combination"},
])
print(_xai_method_matrix.to_string(index=False))
_xai_method_matrix.to_csv(PLOTS_DIR / "v11_xai_method_matrix.csv", index=False)
_log(f"\n  Saved XAI method matrix to {PLOTS_DIR / 'v11_xai_method_matrix.csv'}")
_log(f"\n  All XAI plots saved to: {PLOTS_DIR}")
_log("\n✅  V11 model + XAI notebook complete.")
